In [0]:
%sql
SELECT current_catalog() AS catalog_name, current_schema() AS schema_name;

CREATE VOLUME IF NOT EXISTS YellowTaxiData;

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# Get date 2 months back to ensure data is published
today = datetime.today() - relativedelta(months=3)
year = today.year
month = today.month

# Define URLs and paths
parquet_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
raw_path = f"/Volumes/db_workspace_1_7405609335306068/default/yellowtaxidata/{year}/raw_{month:02d}.parquet"
snappy_path = f"/Volumes/db_workspace_1_7405609335306068/default/yellowtaxidata/{year}/snappy_{month:02d}.parquet"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(raw_path), exist_ok=True)

# Download the file
print(f"Downloading: {parquet_url}")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(parquet_url, headers=headers)

if response.status_code != 200:
    print(f"Failed to download: HTTP {response.status_code}")
elif not response.content[:4] == b'PAR1':
    print(f"URL did not return a valid Parquet file. Data for {year}-{month:02d} may not be published yet.")
else:
    # Save raw file
    with open(raw_path, "wb") as f:
        f.write(response.content)
    print(f"Raw file written to {raw_path}")

    # Re-save as snappy compressed parquet
    print("Converting to snappy compression...")
    df = pd.read_parquet(raw_path)
    df.to_parquet(snappy_path, compression="snappy")
    print(f"Snappy file written to {snappy_path}")

    # Load into Spark and verify
    print("Loading into Spark...")
    df_spark = spark.read.parquet(snappy_path)
    print(f"Row count: {df_spark.count():,}")
    df_spark.printSchema()
    df_spark.show(5)

    

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS bronze
    """)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.yellow_taxi
    USING DELTA
    AS
    SELECT
         *
        ,current_timestamp() AS ingested_at
        ,input_file_name()   AS source_file
        FROM parquet.`{snappy_path}`
        """)



    

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.zone_lookup
    USING DELTA
    AS
    SELECT
         *
        ,current_timestamp() AS ingested_at
        ,input_file_name()   AS source_file
        FROM read_files(
        '/Volumes/db_workspace_1_7405609335306068/default/yellowtaxidata/zone_lookup/taxi_zone_lookup.csv',
        format => 'csv',
        header => true,
        inferSchema => true
    )
        """)

In [0]:
like_pattern = f"%snappy_{month:02d}%"

already_loaded = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM bronze.yellow_taxi 
    WHERE source_file LIKE '{like_pattern}'
    """).collect()[0]['cnt']

print(f"Already loaded: {already_loaded}")

if already_loaded > 0:
    print(f"Data for {year}-{month} already exists, skipping.")
else:
    df_spark = df_spark.withColumn("ingested_at", current_timestamp()) \
           .withColumn("source_file", lit(snappy_path))
    df_spark.write.format("delta") \
        .mode("append") \
        .saveAsTable("bronze.yellow_taxi")
    print(f"Appended {year}-{month} to bronze.yellow_taxi")


In [0]:
%sql
select * from bronze.yellow_taxi

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS silver
    """)

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.yellow_taxi
    USING DELTA
    AS
    SELECT *
    FROM bronze.yellow_taxi
    WHERE 1 = 0
""")

In [0]:
# Load data from bronze to silver
spark.sql(f"""
    INSERT INTO silver.yellow_taxi
    SELECT *
    FROM bronze.yellow_taxi
    WHERE source_file LIKE '{like_pattern}'
""")

# Get the row count separately
load_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM bronze.yellow_taxi
    WHERE source_file LIKE '{like_pattern}'
""").collect()[0]['cnt']

print(f"Rows loaded: {load_count:,}")

In [0]:
spark.sql("ALTER TABLE silver.yellow_taxi ADD COLUMN Valid_Flag VARCHAR(10)")

spark.sql("ALTER TABLE silver.yellow_taxi ADD COLUMN Invalid_Reason VARCHAR(500)")

In [0]:
spark.sql("""
    UPDATE silver.yellow_taxi 
    SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Invalid Trip Distance'
    WHERE trip_distance <= 0
""")

spark.sql("UPDATE silver.yellow_taxi SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Invalid Fare Amount' where fare_amount <= 0")

spark.sql("UPDATE silver.yellow_taxi SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Invalid Passenger Count' where passenger_count <= 0")

spark.sql("UPDATE silver.yellow_taxi SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Invalid Pickup/Dropoff Combination' where tpep_pickup_datetime > tpep_dropoff_datetime")

spark.sql("UPDATE silver.yellow_taxi SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Null PULocationID' where PULocationID is null")

spark.sql("UPDATE silver.yellow_taxi SET Invalid_Reason = COALESCE(Invalid_Reason || ', ', '') || 'Null DOLocationID' where DOLocationID is null")


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import count, col, row_number

df = spark.read.table("silver.yellow_taxi")

# All columns except the two added by us
ignore_cols = ["Valid_Flag", "Invalid_Reason"]
partition_cols = [c for c in df.columns if c not in ignore_cols]

# Use row_number to identify duplicate rows (keep first, flag the rest)
window = Window.partitionBy(partition_cols).orderBy(partition_cols[0])

df = df.withColumn("row_num", row_number().over(window))

# Preview how many duplicates exist
dupe_count = df.filter(col("row_num") > 1).count()
print(f"Duplicate rows found: {dupe_count:,}")

In [0]:
from pyspark.sql.functions import when, coalesce, lit, concat

df = df.withColumn(
    "Invalid_Reason",
    when(
        col("row_num") > 1,
        concat(coalesce(concat(col("Invalid_Reason"), lit(", ")), lit("")), lit("Duplicate"))
    ).otherwise(col("Invalid_Reason"))
).drop("row_num")

In [0]:
df = df.withColumn("der_payment_type",
    when(col("payment_type") == 0, "Flex Fare Trip")
    .when(col("payment_type") == 1, "Credit Card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No Charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 5, "Unknown")
    .when(col("payment_type") == 6, "Voided Trip")
    .otherwise("Other")
)

In [0]:
import pyspark.sql.functions as F

df = df.withColumn(
    "minutes_diff",
    F.round(
        (F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60,
        2
    )
)

In [0]:
df = df.withColumn(
    "mph",
    F.round(
        F.try_divide(
            col("trip_distance"),
            (F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 3600
        ),
        3
    )
)

In [0]:
df = df.withColumn(
    "Valid_Flag",
    F.when(F.col("Invalid_Reason").isNotNull(), "Invalid").otherwise("Valid")
)

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.yellow_taxi")

In [0]:
zone_lookup = spark.table("bronze.zone_lookup")
yellow_taxi_data = spark.table("silver.yellow_taxi")

# Join pickup location
result_pu_df = yellow_taxi_data.join(
    zone_lookup,
    on=yellow_taxi_data["PULocationID"] == zone_lookup["LocationID"],
    how="left"
).select(
    yellow_taxi_data["*"],                  # all columns from left
    zone_lookup["Borough"].alias("PUBorough"),         # specific columns from right
    zone_lookup["Zone"].alias("PUZone"),
    zone_lookup["Service_Zone"].alias("PUService_Zone"))

    # Join pickup location
result_bothlocations_df = result_pu_df.join(
    zone_lookup,
    on=result_pu_df["DOLocationID"] == zone_lookup["LocationID"],
    how="left"
).select(
    result_pu_df["*"],                  # all columns from left
    zone_lookup["Borough"].alias("DOBorough"),         # specific columns from right
    zone_lookup["Zone"].alias("DOZone"),
    zone_lookup["Service_Zone"].alias("DOService_Zone"))


result_bothlocations_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.yellow_taxi")




In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS gold
    """)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE gold.yellow_taxi_daily_revenue_by_payment_type
    USING DELTA
    COMMENT 'Daily Revenue by payment type'
    AS
    SELECT
        DATE(t.tpep_pickup_datetime)                         AS trip_date,
        ROUND(SUM(t.total_amount),2) AS total_revenue,
        t.der_payment_type                            AS payment_type
    FROM silver.yellow_taxi t
    WHERE t.tpep_pickup_datetime IS NOT NULL
    AND t.Valid_Flag='Valid'
    GROUP BY
        DATE(t.tpep_pickup_datetime), der_payment_type 
    ORDER BY DATE(t.tpep_pickup_datetime), t.der_payment_type
    """)



In [0]:
%sql
select * from gold.yellow_taxi_daily_revenue_by_payment_type

In [0]:
spark.sql("OPTIMIZE gold.yellow_taxi_daily_revenue_by_payment_type  ZORDER BY (trip_date)")

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE gold.yellow_taxi_total_trips_started_from_borough_vs_ending_at_borough_by_date
    USING DELTA
    AS
    select x.*, y.Total_Trips_Ending_In_Borough, y.DOBorough from (select count(*) as Total_Trips_Starting_In_Borough, PUBorough, DATE(s.tpep_pickup_datetime) as dt from silver.yellow_taxi s
where s.Valid_Flag = 'Valid'
and s.tpep_pickup_datetime is not null
and s.tpep_dropoff_datetime is not null
group by PUBorough, DATE(s.tpep_pickup_datetime)
order by DATE(s.tpep_pickup_datetime), PUBorough) x
full outer join  (select count(*) as Total_Trips_Ending_In_Borough, DOBorough, DATE(s.tpep_dropoff_datetime) as dt from silver.yellow_taxi s
where s.Valid_Flag = 'Valid'
and s.tpep_dropoff_datetime is not null
and s.tpep_pickup_datetime is not null
group by DOBorough, DATE(s.tpep_dropoff_datetime)
order by DATE(s.tpep_dropoff_datetime), DOBorough) y
on x.PUBorough = y.DOBorough
and x.dt = y.dt
order by (x.dt, x.PUBorough)
""")

In [0]:
%sql
select * from gold.yellow_taxi_total_trips_started_from_borough_vs_ending_at_borough_by_date